# Khánh — calibration và evaluation của model đã khóa

- Dùng model bàn giao của Duy, không chọn lại bằng test.
- Bản LightGBM trước sửa được giữ ở `archive/02_steel_modeling_evaluation_2026-09-30.ipynb`; kết quả đó không thuộc HGB hiện tại.
- Mặc định **không chạy calibration/test**. Khánh và nhóm cần chốt protocol rồi bật từng công tắc.
- Threshold là p95 tổng kWh giờ tới của TRAIN; buffer là phân vị thực nghiệm lỗi dự báo thiếu trên tháng 10. Đây chưa phải P90 có bảo đảm hoặc giới hạn công suất hợp đồng.
- Test tháng 11–12 đã được xem ở các lần làm trước, nên không gọi lần sau là blind test.


In [1]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if not (ROOT / "src/steel").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/steel").is_dir():
    raise FileNotFoundError("Mở notebook từ repo hoặc thư mục notebooks")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from src.steel.duy_inference import DuyForecaster
from src.steel.duy_forecasting import baseline_predictions, predict_energy
from src.steel.evaluation import calibrate_upper_buffer, evaluate_alerts
from src.steel.preprocessing import BASE_FEATURES, TARGET, load_official_raw

RUN_DIR = ROOT / "reports/steel/duy_2026-10-02"
OUTPUT_DIR = ROOT / "reports/steel/khanh_evaluation_2026-10-02"
RUN_CALIBRATION = False
RUN_TEST = False

forecaster = DuyForecaster(RUN_DIR)
manifest = forecaster.manifest
DATA_DIR = RUN_DIR / "processed"
summary = json.loads((DATA_DIR / "steel_quality_summary.json").read_text(encoding="utf-8"))
if summary["base_feature_columns"] != list(BASE_FEATURES):
    raise ValueError("Schema không khớp; không dùng feature fallback")
if summary["feature_implementation"] != manifest["feature_implementation"]:
    raise ValueError("Processed/model khác cách tính feature")
if summary["source_sha256"] != manifest["source_sha256"]:
    raise ValueError("Processed/model khác nguồn")
THRESHOLD = summary["train_target_p95_kWh"]
print("Model đã khóa:", manifest["selected_model"])
print("Calibration:", RUN_CALIBRATION, "| Test:", RUN_TEST)
print("Không có calibration/test score mới khi hai công tắc là False.")


Model đã khóa: hgb_31_leaves
Calibration: False | Test: False
Không có calibration/test score mới khi hai công tắc là False.


In [2]:
TIME_COLUMNS = ["observation_time", "forecast_start", "forecast_end"]

def load_part(name):
    return pd.read_csv(DATA_DIR / f"steel_next_60m_{name}.csv",
                       parse_dates=TIME_COLUMNS, float_precision="round_trip")

def predict_part(part):
    with threadpool_limits(limits=1):
        if manifest["kind"] == "AI":
            return predict_energy(forecaster.model, part[list(BASE_FEATURES)])
        # Giữ baseline nếu nó thắng; diurnal chỉ học TRAIN và đúng giờ timestamp.
        raw, _ = load_official_raw(ROOT / "data/steel/raw/Steel_industry_data.csv")
        return baseline_predictions(raw, load_part("train"), part)[manifest["selected_model"]]

policy = None
if RUN_CALIBRATION:
    calibration = load_part("calibration")
    calibration_prediction = predict_part(calibration)
    buffer = calibrate_upper_buffer(calibration[TARGET], calibration_prediction, .90)
    policy = {
        "selected_model": manifest["selected_model"],
        "model_sha256": manifest["model_sha256"],
        "baseline_state_sha256": manifest["baseline_state_sha256"],
        "source_sha256": manifest["source_sha256"],
        "features": list(BASE_FEATURES),
        "feature_implementation": manifest["feature_implementation"],
        "target": TARGET, "threshold_train_p95_kWh": THRESHOLD,
        "buffer_kWh": buffer, "residual_quantile": .90,
        "calibration_period": "October 2018",
        "rule": "prediction_kWh + buffer_kWh > threshold_train_p95_kWh",
        "old_test_already_observed": True,
        "coverage_guaranteed": False, "demand_limit": False,
    }
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with (OUTPUT_DIR / "alert_policy.json").open("x", encoding="utf-8") as file:
        json.dump(policy, file, ensure_ascii=False, indent=2)
    print("Đã khóa policy cho đúng model:", policy)
else:
    print("Chưa chạy calibration; chưa có policy mới.")


Chưa chạy calibration; chưa có policy mới.


## Đánh giá test sau khi khóa model và policy

- Không dùng MAE test để đổi model/ngưỡng/buffer. Nếu cần đổi, ghi thành vòng phát triển mới.
- MAE/RMSE cho hồi quy; confusion matrix, precision/recall/F1, FPR và false discovery rate cho cảnh báo.
- Cụm cảnh báo là các mốc phát dự báo liên tiếp trên lưới 15 phút; không nối qua gap.
- Episode overlap là metric nhiều–nhiều; không diễn giải thành cảnh báo trước đủ 60 phút.
- Độ trễ trong bảng validation Duy là model-only. Huy phải đo thêm đường Data → Feature → Model → Policy → UI.


In [3]:
if RUN_TEST:
    if policy is None:
        raise ValueError("Chạy và khóa calibration trước test; không tự dùng policy cũ")
    test = load_part("test")
    test_prediction = predict_part(test)
    results = evaluate_alerts(test[TARGET], test_prediction, test["observation_time"],
                              policy["threshold_train_p95_kWh"], policy["buffer_kWh"])
    results.update({"selected_model": manifest["selected_model"],
                    "model_sha256": manifest["model_sha256"],
                    "old_test_already_observed": True})
    with (OUTPUT_DIR / "test_results.json").open("x", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False, indent=2)
    display(pd.Series(results))
else:
    print("Test chưa được chấm điểm trong notebook này.")


Test chưa được chấm điểm trong notebook này.


## Việc Khánh/Huy cần hoàn thiện

- Khánh: xem false negatives/false alarms, metric theo tháng/giờ/tải cao; kiểm drift và bất định với dữ liệu chuỗi có phụ thuộc.
- Huy: UI tình trạng đầu vào, dự báo, trạng thái policy, cảnh báo, khuyến nghị kiểm tra và người duyệt; log quyết định, theo dõi lỗi có nhãn trễ.
- Kịch bản công nghiệp chỉ là mô phỏng với giả định được ghi rõ; chưa có hành động thật để đo tiết kiệm, downtime hoặc năng suất.
- Kiểm rubric và đọc `docs/team/review_and_release_2026-10-02.md` trước tích hợp.
